In [53]:
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully.")

Libraries imported successfully.


In [54]:
df = pd.read_csv("../data/energydata_complete.csv")

print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:", df.duplicated().sum())

display(df.head())

Dataset shape: (19735, 29)

Missing values:
0

Duplicate rows: 0


,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


In [55]:
# Exploratory Data Analysis

df["date"] = pd.to_datetime(df["date"])

print("Date range:")
print(df["date"].min(), "→", df["date"].max())

print("\nTarget statistics:")
display(df["Appliances"].describe())

print("\nTop 10 highest consumption records:")
display(
    df.nlargest(10, "Appliances")[["date", "Appliances"]]
)

Date range:
2016-01-11 17:00:00 → 2016-05-27 18:00:00

Target statistics:


count    19735.000000
mean        97.694958
std        102.524891
min         10.000000
25%         50.000000
50%         60.000000
75%        100.000000
max       1080.000000
Name: Appliances, dtype: float64


Top 10 highest consumption records:


,date,Appliances
731,2016-01-16 18:50:00,1080
1451,2016-01-21 18:50:00,1070
432,2016-01-14 17:00:00,910
12088,2016-04-04 15:40:00,900
1452,2016-01-21 19:00:00,890
10668,2016-03-25 19:00:00,880
14693,2016-04-22 17:50:00,870
9031,2016-03-14 10:10:00,860
1821,2016-01-24 08:30:00,850
19582,2016-05-26 16:40:00,850


In [56]:
print("Correlation with Appliances:")

correlation = (
    df.select_dtypes(include=np.number)
      .corr()["Appliances"]
      .sort_values(ascending=False)
)

display(correlation)

Correlation with Appliances:


Appliances     1.000000
lights         0.197278
T2             0.120073
T6             0.117638
T_out          0.099155
Windspeed      0.087122
RH_1           0.086031
T3             0.085060
T1             0.055447
T4             0.040281
T8             0.039572
RH_3           0.036292
T7             0.025801
T5             0.019760
RH_4           0.016965
Tdewpoint      0.015353
T9             0.010010
RH_5           0.006955
Visibility     0.000230
rv1           -0.011145
rv2           -0.011145
Press_mm_hg   -0.034885
RH_9          -0.051462
RH_7          -0.055642
RH_2          -0.060465
RH_6          -0.083178
RH_8          -0.094039
RH_out        -0.152282
Name: Appliances, dtype: float64

In [57]:
# Create time-series forecasting features

df_ts = df.copy()

# Ensure chronological order
df_ts = df_ts.sort_values("date").reset_index(drop=True)

# Time features
df_ts["hour"] = df_ts["date"].dt.hour
df_ts["day_of_week"] = df_ts["date"].dt.dayofweek
df_ts["month"] = df_ts["date"].dt.month
df_ts["is_weekend"] = (df_ts["day_of_week"] >= 5).astype(int)

# Previous energy consumption
df_ts["Appliances_lag_1"] = df_ts["Appliances"].shift(1)
df_ts["Appliances_lag_2"] = df_ts["Appliances"].shift(2)
df_ts["Appliances_lag_3"] = df_ts["Appliances"].shift(3)

# Recent average consumption
df_ts["Appliances_rolling_mean_3"] = (
    df_ts["Appliances"]
    .shift(1)
    .rolling(3)
    .mean()
)

df_ts = df_ts.dropna().reset_index(drop=True)

print("Shape after lag features:", df_ts.shape)

Shape after lag features: (19732, 37)


In [58]:
# Prepare features and target

X = df_ts.drop(columns=["Appliances", "date"])
y = df_ts["Appliances"]

split = int(len(df_ts) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (15785, 35)
Testing : (3947, 35)


In [59]:
# Naive forecast: assume next value ≈ previous value
baseline_pred = X_test["Appliances_lag_1"]

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("NAIVE BASELINE")
print(f"MAE  : {baseline_mae:.2f} Wh")
print(f"RMSE : {baseline_rmse:.2f} Wh")
print(f"R²   : {baseline_r2:.3f}")

NAIVE BASELINE
MAE  : 26.65 Wh
RMSE : 66.44 Wh
R²   : 0.467


In [60]:
model = RandomForestRegressor(
    n_estimators=150,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("RANDOM FOREST MODEL")
print(f"MAE  : {mae:.2f} Wh")
print(f"RMSE : {rmse:.2f} Wh")
print(f"R²   : {r2:.3f}")

RANDOM FOREST MODEL
MAE  : 62.38 Wh
RMSE : 95.70 Wh
R²   : -0.105


In [61]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Top 10 predictive features:")
display(importance.head(10))

Top 10 predictive features:


Appliances_lag_1             0.641053
Appliances_lag_2             0.037224
Appliances_rolling_mean_3    0.019502
hour                         0.016622
Appliances_lag_3             0.014137
T3                           0.013678
Press_mm_hg                  0.012707
T4                           0.012497
T8                           0.011851
RH_1                         0.011756
dtype: float64

In [62]:
import os
import joblib

os.makedirs("../model", exist_ok=True)

joblib.dump(
    {
        "model": model,
        "features": X.columns.tolist()
    },
    "../model/energy_model.pkl"
)

print("Model + feature order saved.")
print("Features:", X.columns.tolist())

Model + feature order saved.
Features: ['lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'rv1', 'rv2', 'hour', 'day_of_week', 'month', 'is_weekend', 'Appliances_lag_1', 'Appliances_lag_2', 'Appliances_lag_3', 'Appliances_rolling_mean_3']
